# Regression Foundation & Causal Toolbox (Detailed): GLM, Multinomial/Ordinal, Marginal Effects, Instrumental Variables, Matching, Mediation

Earlier tutorials covered specialized methods (DiD, RDD, SEM, survival...). This one fills in the **most routine** commands in social science—generalized regression, multinomial and ordinal outcomes, marginal effects, instrumental variables, propensity score matching, and causal mediation. For R users they are `glm`/`multinom`/`polr`/`margins`/`ivreg`/`MatchIt`/`mediation`; for Stata users, `logit`/`mlogit`/`ologit`/`margins`/`ivregress`/`psmatch2`/`mediate`; nearly every empirical paper uses them.

This tutorial is a **detailed exposition**: each method is run in full, showing complete coefficient tables (point estimates, standard errors, 95% confidence intervals, z values, p values, fit statistics), explaining how to interpret them, and comparing against **known true values** in built-in synthetic data to prove these implementations are not placeholders. Each section begins by marking the corresponding R / Stata / SPSS commands for easy migration.

All functions are registered in the socialverse registry and operate on the same `StudyState`. If you come from R/Stata/SPSS, you can look them up directly by command name: `sv.registry.get("py-logit")`, `py-ivregress`, `py-psmatch2`, `py-mediate` all resolve to the corresponding functions.

In [1]:
import numpy as np
import pandas as pd

import socialverse as sv
from socialverse import datasets as ds

pd.set_option("display.float_format", lambda v: f"{v:.4f}")


def coef_table(model, drop_const=False):
    """把一个存了 coef/se/ci/z/p 的模型 dict 渲染成完整系数表(DataFrame)。"""
    rows = []
    for name, b in model["coef"].items():
        if drop_const and name == "const":
            continue
        se = model.get("se", {}).get(name)
        ci = model.get("ci", {}).get(name)
        z = (model.get("z") or model.get("t") or {}).get(name)
        p = model.get("p", {}).get(name)
        rows.append({
            "变量": name, "系数": b, "标准误": se,
            "CI下": ci[0] if ci else None, "CI上": ci[1] if ci else None,
            "z": z, "p": p,
        })
    return pd.DataFrame(rows).set_index("变量")

## 1. Generalized Linear Regression `glm` — One function covers OLS / logit / probit / Poisson

**Analogous to**: R `stats::glm` · Stata `regress`/`logit`/`poisson` · SPSS `REGRESSION`/`GENLIN`.

Social science outcome variables come in various forms: continuous (income), binary (whether to vote), count (number of protests). Generalized linear models unify them using a `family` (error distribution) + link function: `gaussian` is ordinary least squares (OLS), `binomial` is logit/probit, `poisson` is a count model. The socialverse `glm` is a single entry point, supporting robust (HC1) and clustered standard errors.

Synthetic data `load_regression` embeds known true values: for continuous `y`, coefficients x1=0.5, x2=−0.4; for binary `y_bin`, logit true values x1=0.8, x2=−0.5; for count `y_count`, Poisson true value x1=0.4.

In [2]:
df = ds.load_regression()
print("数据形状:", df.shape, "| 列:", list(df.columns))
df.head()

数据形状: (600, 7) | 列: ['x1', 'x2', 'y', 'y_bin', 'y_count', 'y_ord', 'choice']


,x1,x2,y,y_bin,y_count,y_ord,choice
0,0.1257,-1.1568,0.1988,1,3,1,B
1,-0.1321,-1.3649,1.7838,1,2,2,A
2,0.6404,-0.2312,2.7605,1,1,2,B
3,0.1049,2.2775,-0.2227,0,0,1,C
4,-0.5357,0.2775,-0.6474,0,2,2,C


### 1a. OLS (`family="gaussian"`)

The most basic linear regression. Coefficients directly mean "how much y changes on average when x increases by one unit".

In [3]:
st = sv.StudyState()
sv.pp.ingest(st, data=df)
st.write("variables", "outcome", "y")
sv.tl.glm(st, predictors=["x1", "x2"], family="gaussian", cov="robust")

m = st.models["glm"]
print(f"估计量: {m['estimator']} · 标准误: {m['cov']} · n = {m['n']}")
print(f"拟合: R² = {st.diagnostics['glm_fit'].get('r2')}, AIC = {st.diagnostics['glm_fit']['aic']:.1f}")
print("真值: x1 = 0.5, x2 = -0.4\n")
coef_table(m)

估计量: statsmodels.GLM(gaussian) · HC1 robust · 标准误: robust · n = 600
拟合: R² = 0.2630884139883549, AIC = 1744.6
真值: x1 = 0.5, x2 = -0.4



,系数,标准误,CI下,CI上,z,p
变量,,,,,,
const,0.9922,0.0422,0.9095,1.0749,23.5077,0.0000
x1,0.4487,0.0413,0.3677,0.5297,10.8621,0.0000
x2,-0.4672,0.0447,-0.5548,-0.3796,-10.4559,0.0000


Reading the table: the coefficient for `x1` falls near the true value 0.5, `x2` near −0.4, and 95% confidence intervals exclude 0 for both (p values are negligible). Robust standard errors (HC1) accommodate heteroskedasticity.

### 1b. Logit (`family="binomial"`)

Binary outcomes. **Note**: logit coefficients are on the log-odds scale and cannot be read directly as probability changes—you can see the sign and significance, but the magnitude requires marginal effects (covered in the next section) for interpretation.

In [4]:
st_logit = sv.StudyState()
sv.pp.ingest(st_logit, data=df)
st_logit.write("variables", "outcome", "y_bin")
sv.tl.glm(st_logit, predictors=["x1", "x2"], family="binomial")

ml = st_logit.models["glm"]
print(f"family = {ml['family']} · McFadden pseudo-R² = {st_logit.diagnostics['glm_fit']['pseudo_r2']:.3f}")
print("真值(对数几率尺度): x1 = 0.8, x2 = -0.5\n")
coef_table(ml)

family = binomial · McFadden pseudo-R² = 0.169
真值(对数几率尺度): x1 = 0.8, x2 = -0.5



,系数,标准误,CI下,CI上,z,p
变量,,,,,,
const,-0.3336,0.0938,-0.5174,-0.1499,-3.5584,0.0004
x1,0.9801,0.1105,0.7634,1.1967,8.8682,0.0000
x2,-0.7179,0.1063,-0.9263,-0.5096,-6.7541,0.0000


### 1c. Poisson (`family="poisson"`)

Count outcomes. Coefficients are on the log-rate scale: `exp(coefficient)` is the incidence-rate ratio. True value x1=0.4 means a one-unit increase in x1 multiplies the event rate by `exp(0.4)≈1.49`.

In [5]:
st_pois = sv.StudyState()
sv.pp.ingest(st_pois, data=df)
st_pois.write("variables", "outcome", "y_count")
sv.tl.glm(st_pois, predictors=["x1", "x2"], family="poisson")

mp = st_pois.models["glm"]
tbl = coef_table(mp, drop_const=True)
tbl["发生率比 exp(β)"] = np.exp(tbl["系数"])
print("真值: x1 = 0.4  → exp(0.4) ≈ 1.49\n")
tbl

真值: x1 = 0.4  → exp(0.4) ≈ 1.49



,系数,标准误,CI下,CI上,z,p,发生率比 exp(β)
变量,,,,,,,
x1,0.3585,0.0363,0.2874,0.4295,9.8869,0.0000,1.4312
x2,-0.1422,0.0373,-0.2153,-0.0690,-3.8077,0.0001,0.8675


## 2. Average Marginal Effects `margins` — Translate nonlinear coefficients into interpretable effects

**Analogous to**: Stata `margins` · R `marginaleffects::slopes` / `emmeans`.

Logit/Poisson coefficients are not on the natural scale of the outcome. `margins` computes average marginal effects (AME) **on the just-fitted model**: it calculates the partial effect for each observation, then averages. For logit, the AME is "how many percentage points the probability of the outcome changes on average when x increases by one unit"—this is what you report in a paper. It reads the fitted object stored by the previous `glm` call in the state.

In [6]:
sv.tl.margins(st_logit, model="glm")   # 在 1b 的 logit 上算
mg = st_logit.diagnostics["margins"]
print(f"模型: {mg['model']} · 求值点: {mg['at']}")
pd.DataFrame({"平均边际效应(概率尺度)": mg["ame"]})

模型: glm · 求值点: overall


,平均边际效应(概率尺度)
x1,0.1900
x2,-0.1392


Reading the table: x1's AME is positive, x2's negative—same sign as the logit coefficients, but now on a **probability** scale (e.g., +0.19 means a one-unit increase in x1 raises the event probability by about 19 percentage points on average).

## 3. Multinomial Logit `mlogit` — Unordered Multiclass Outcomes

**Analogous to**: Stata `mlogit` · R `nnet::multinom` · SPSS `NOMREG`.

Outcomes consist of three or more **unordered** categories (e.g., choosing scheme A/B/C). Multinomial logit uses one category as a baseline (base) reference and reports log-odds coefficients for each other category relative to the baseline. In the synthetic data, the tendency to choose B increases with x1, while the tendency to choose C decreases with x1.

In [7]:
st_ml = sv.StudyState()
sv.pp.ingest(st_ml, data=df)
st_ml.write("variables", "outcome", "choice")
sv.tl.mlogit(st_ml, predictors=["x1"])

mm = st_ml.models["mlogit"]
print(f"基准类别: {mm['base']} · 类别: {mm['categories']} · n = {mm['n']} · logL = {mm['llf']:.1f}\n")
rows = []
for cat, params in mm["coef"].items():
    for var, cell in params.items():
        rows.append({"类别(vs %s)" % mm["base"]: cat, "变量": var,
                     "系数(log-odds)": cell["coef"], "标准误": cell["se"]})
pd.DataFrame(rows).set_index(["类别(vs %s)" % mm["base"], "变量"])

基准类别: A · 类别: ['A', 'B', 'C'] · n = 600 · logL = -578.1



系数(log-odds)    标准误
类别(vs A) 变量                        
B        const        0.0113 0.1159
         x1           0.9367 0.1333
C        const        0.1630 0.1101
         x1          -0.5015 0.1213

Reading the table: the `x1` coefficient for B relative to A is **positive** (larger x1 favors B), and for C relative to A is **negative** (larger x1 disfavors C)—consistent with the data-generating process.

## 4. Ordinal Logit `ologit` — Ordered Multiclass Outcomes

**Analogous to**: Stata `ologit`/`oprobit` · R `MASS::polr` · SPSS `PLUM`.

Outcomes consist of **ordered** categories (e.g., "disagree < neutral < agree"). Ordinal logit assumes an underlying latent variable cut by thresholds (cutpoints) into ordered segments; it estimates a single set of slope coefficients plus the cutpoints. A positive coefficient means a larger predictor makes falling into a higher category more likely.

In [8]:
st_ol = sv.StudyState()
sv.pp.ingest(st_ol, data=df)
st_ol.write("variables", "outcome", "y_ord")
sv.tl.ologit(st_ol, predictors=["x1", "x2"], link="logit")

mo = st_ol.models["ologit"]
print(f"{mo['estimator']} · {mo['n_categories']} 个有序等级 · n = {mo['n']}")
print("切点(thresholds):", {k: round(v, 3) for k, v in mo["thresholds"].items()}, "\n")
pd.DataFrame({"系数(潜变量尺度)": mo["coef"], "标准误": mo["coef_se"]})

statsmodels.OrderedModel(logit) · 3 个有序等级 · n = 600
切点(thresholds): {'cut_1': -0.519, 'cut_2': 0.718} 



,系数(潜变量尺度),标准误
x1,0.6194,0.0836
x2,-0.5313,0.0868


Reading the table: the `x1` coefficient is positive (larger x1 predicts higher ordered categories), `x2` is negative. Two cutpoints divide the latent variable into 3 segments.

## 5. Instrumental Variables `iv_regress` — When a Regressor Is Endogenous

**Analogous to**: Stata `ivregress`/`ivreg2` · R `AER::ivreg`/`fixest` · SPSS `2SLS`.

If a key regressor correlates with the error term (omitted variables, reverse causality, measurement error), OLS is biased. Instrumental variables use an "instrument" that affects the outcome only through the regressor to isolate endogeneity: two-stage least squares (2SLS) first predicts the endogenous variable using the instrument, then estimates the effect with the prediction, computing standard errors with the **correct 2SLS covariance**.

In synthetic data `load_iv`, `x` is contaminated by unobserved confounding; the true causal effect = 1.5; OLS would overestimate; instrument `z` helps us recover it. **Weak instrument test**: the first-stage F must be well above 10 to be credible.

In [9]:
st_iv = sv.StudyState()
sv.pp.ingest(st_iv, data=ds.load_iv())
st_iv.write("variables", "outcome", "y")
sv.tl.iv_regress(st_iv, endogenous="x", instruments=["z"], exog=["w"])

miv = st_iv.models["iv"]
print(f"{miv['estimator']} · 标准误: {miv['cov_type']} · n = {miv['n']}")
print("真值: x 的因果效应 = 1.5\n")
coef_table(miv)

2SLS (hand-rolled projection) · 标准误: robust(HC0) · n = 800
真值: x 的因果效应 = 1.5



,系数,标准误,CI下,CI上,z,p
变量,,,,,,
const,0.9106,0.0820,0.7498,1.0713,None,0.0000
x,1.3683,0.1259,1.1215,1.6150,None,0.0000
w,0.4989,0.0929,0.3168,0.6809,None,0.0000


In [10]:
fs = st_iv.diagnostics["first_stage"]
pd.DataFrame({
    "指标": ["一阶段 F(排除性工具联合显著)", "是否弱工具(F<10)",
             "OLS 内生系数(有偏)", "2SLS 内生系数(修正)"],
    "值": [round(fs["F"], 1), fs["weak_instrument"],
           round(fs["ols_endog_coef"], 3), round(fs["iv_endog_coef"], 3)],
}).set_index("指标")

,值
指标,
一阶段 F(排除性工具联合显著),624.1000
是否弱工具(F<10),False
OLS 内生系数(有偏),2.5720
2SLS 内生系数(修正),1.3680


Reading the table: 2SLS estimates the effect of `x` at ≈1.37 (true value 1.5), while **biased OLS overestimates to ≈2.57**; first-stage F=624 ≫ 10, so the instrument is strong, not weak. This is exactly the scenario where naive OLS misleads but instrumental variables correct the bias.

## 6. Propensity Score Matching `psm` — Estimating Treatment Effects from Observational Data

**Analogous to**: Stata `teffects psmatch`/`psmatch2` · R `MatchIt`/`WeightIt`.

Without random assignment, treatment and control groups typically differ on covariates. Propensity score matching first estimates the "probability of receiving treatment" from covariates, then pairs treated units with control units of similar propensity (or uses inverse-probability weighting), recovering the average treatment effect on the treated (ATT) from the comparable subsample. In synthetic data `load_treatment`, treatment is driven by x1..x3 (selection bias); true ATT=2.0; naive group difference is biased.

In [11]:
st_psm = sv.StudyState()
sv.pp.ingest(st_psm, data=ds.load_treatment())
st_psm.write("variables", "outcome", "y")
st_psm.write("design", "treatment", "treat")

sv.tl.psm(st_psm, covariates=["x1", "x2", "x3"], method="nn")   # 最近邻 1:1
nn = st_psm.models["psm"]
sv.tl.psm(st_psm, covariates=["x1", "x2", "x3"], method="ipw")  # 逆概率加权
ipw = st_psm.models["psm"]

pd.DataFrame({
    "估计": ["ATT(最近邻匹配)", "ATT(逆概率加权 IPW)", "朴素差值(未调整)"],
    "值": [nn["att"], ipw["att"], nn["naive_diff"]],
    "标准误": [nn["se"], ipw["se"], None],
}).set_index("估计")

,值,标准误
估计,,
ATT(最近邻匹配),1.7608,0.1207
ATT(逆概率加权 IPW),2.1047,0.1230
朴素差值(未调整),2.6700,NaN


The point of matching is to make covariates comparable across groups. Diagnostics show **standardized mean differences (SMD)** before and after matching; after matching it should drop substantially (empirically, |SMD|<0.1 counts as good balance):

In [12]:
bal = st_psm.diagnostics["balance"]
pd.DataFrame({
    "协变量": list(bal["smd_before"]),
    "匹配前 SMD": [float(bal["smd_before"][c]) for c in bal["smd_before"]],
    "匹配后 SMD": [float(bal["smd_after"][c]) for c in bal["smd_before"]],
}).set_index("协变量")

,匹配前 SMD,匹配后 SMD
协变量,,
x1,0.8285,0.0382
x2,-0.3444,-0.0527
x3,0.2054,-0.0026


Reading the table: matched/weighted ATT estimates cluster near the true value 2.0 (nearest-neighbor slightly low, IPW slightly high), while **naive difference 2.67 is markedly high** (inflated by selection bias). After matching, x1's SMD drops from 0.83 to nearly 0—covariates have been balanced.

## 7. Causal Mediation `mediation` — Through What Channel Does an Effect Operate?

**Analogous to**: Stata `mediate`/`med4way` · R `mediation::mediate` · SPSS PROCESS (Hayes).

Knowing that X affects Y is not enough; you often want to ask "how much of this effect travels **through** mediator M?" Mediation analysis decomposes the total effect into: the indirect effect passing through M (ACME) and the direct effect bypassing M (ADE). socialverse fits a mediation model (x→m, extracting coefficient a) and an outcome model (y→x+m, extracting b and direct effect c'), estimates ACME using the product a·b, and provides confidence intervals via **nonparametric bootstrap** (cross-validated optionally using statsmodels' Mediation class).

Synthetic data `load_mediation` has a=0.6, b=0.7 → indirect ACME=0.42, direct ADE=0.30, total 0.72.

In [13]:
st_med = sv.StudyState()
sv.pp.ingest(st_med, data=ds.load_mediation())
st_med.write("variables", "outcome", "y")
sv.tl.mediation(st_med, treatment="x", mediator="m", boot=1000, seed=0)

med = st_med.models["mediation"]
print(f"路径: a(x→m)={med['a']:.3f}, b(m→y)={med['b']:.3f}, 直接 c'={med['direct']:.3f}")
print(f"bootstrap: {med['boot']} 次 · 交叉校验({med['crosscheck']['source'].split('.')[-1]}): ACME={med['crosscheck']['acme']:.3f}\n")
pd.DataFrame({
    "效应": ["间接 ACME (a·b)", "直接 ADE (c')", "总效应", "中介占比"],
    "估计": [med["acme"], med["ade"], med["total"], med["prop_mediated"]],
    "95%CI下": [med["ci_acme"][0], med["ci_ade"][0], med["ci_total"][0], None],
    "95%CI上": [med["ci_acme"][1], med["ci_ade"][1], med["ci_total"][1], None],
    "真值": [0.42, 0.30, 0.72, 0.42 / 0.72],
}).set_index("效应")

路径: a(x→m)=0.652, b(m→y)=0.633, 直接 c'=0.289
bootstrap: 1000 次 · 交叉校验(Mediation): ACME=0.411



,估计,95%CI下,95%CI上,真值
效应,,,,
间接 ACME (a·b),0.4124,0.3340,0.4856,0.4200
直接 ADE (c'),0.2890,0.1905,0.3915,0.3000
总效应,0.7015,0.6062,0.7873,0.7200
中介占比,0.5879,NaN,NaN,0.5833


Reading the table: indirect effect ACME≈0.41 (true value 0.42), direct ADE≈0.29 (true value 0.30), total≈0.70 (true value 0.72); bootstrap confidence intervals exclude zero; approximately 59% of the total effect operates through mediator M. Cross-validation with the statsmodels Mediation implementation yields nearly identical ACME, confirming correctness.

## Summary

This tutorial details the most frequently used quantitative foundations in social science: generalized regression (`glm` covering OLS/logit/probit/poisson), multinomial and ordinal outcomes (`mlogit`/`ologit`), marginal effects (`margins`), instrumental variables (`iv_regress`), propensity score matching (`psm`), and causal mediation (`mediation`). Each includes complete coefficient tables, diagnostics, and validation against known true values in the data—these are not placeholder implementations. Together with the specialized methods covered earlier, socialverse now spans the main trunk from "basic regression" to "cutting-edge quasi-experiments".

Migration guide: they align with R's `glm`/`multinom`/`polr`/`margins`/`ivreg`/`MatchIt`/`mediation` and Stata's `logit`/`mlogit`/`ologit`/`margins`/`ivregress`/`psmatch2`/`mediate`. Look them up by familiar command name:

In [14]:
for cmd in ["py-logit", "py-mlogit", "py-polr", "py-margins", "py-ivregress", "py-psmatch2", "py-mediate"]:
    e = sv.registry.get(cmd)
    print(f"  {cmd:14s} -> sv.tl.{e.name}" if e else f"  {cmd}: (未命中)")

  py-logit       -> sv.tl.glm
  py-mlogit      -> sv.tl.mlogit
  py-polr        -> sv.tl.ologit
  py-margins     -> sv.tl.margins
  py-ivregress   -> sv.tl.iv_regress
  py-psmatch2    -> sv.tl.psm
  py-mediate     -> sv.tl.mediation
